# JME Call for Papers Revision

In [1]:
import glob, os, json, time, random, pickle, math, re
import pandas as pd
import numpy as np
import torch

import matplotlib.pyplot as plt
import cv2, PIL, base64
from moviepy import VideoFileClip

from tqdm import tqdm, trange
from functools import partial
from pathlib import Path
from collections import defaultdict

import utils, data_utils

os.environ['KMP_DUPLICATE_LIB_OK']='True'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Additional Data Collection

In [9]:
url = "https://www.youtube.com/watch?v=dGNAwxi9uwA"
download_dir = "data/youtube"
# data_utils.download_video(url, download_dir, download_sections=(12,21))
data_utils.download_video(url, download_dir)

[youtube] Extracting URL: https://www.youtube.com/watch?v=dGNAwxi9uwA
[youtube] dGNAwxi9uwA: Downloading webpage


[youtube] dGNAwxi9uwA: Downloading visionos player API JSON
[youtube] dGNAwxi9uwA: Downloading m3u8 information
[info] dGNAwxi9uwA: Downloading 1 format(s): 616
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 99
[download] Destination: data\youtube\How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4
[download] 100% of  283.04MiB in 00:00:37 at 7.50MiB/s                  


'data/youtube\\How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4'

In [10]:
source_dir = "data/youtube"
target_dir = "data/new_clipped"

source_video_filename = "How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout" + ".mp4"
source_video_path = os.path.join(source_dir, source_video_filename)

start_time = 2*60 + 26
end_time = 3*60 + 6
target_video_path = os.path.join(target_dir, f"clipped_{start_time}_{end_time}_{source_video_filename}")

data_utils.clip_video(source_video_path, start_time, end_time, target_video_path)

MoviePy - Building video data/new_clipped\clipped_146_186_How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4.
MoviePy - Writing video data/new_clipped\clipped_146_186_How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4



MoviePy - Done !
MoviePy - video ready data/new_clipped\clipped_146_186_How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4


In [12]:
video_paths = "data/new_clipped/*.mp4"
video_paths = glob.glob(video_paths)
for video_path in video_paths:
    v_path = Path(video_path)
    data_utils.extract_frames(v_path, target_dir_fps1 = Path("data/new_frames_fps1"), target_dir_fps8 = Path("data/new_frames_fps8"))

In [18]:
frame1_dir = "data/new_frames_fps1"
frame_paths = glob.glob(os.path.join(frame1_dir, "*.jpg"))

def sort_key(path):
    filename = os.path.splitext(os.path.basename(path))[0]
    parts = filename.rsplit("_", 2)

    base_filename = parts[0]
    second = int(parts[1])
    frame_idx = int(parts[2])

    return base_filename, second, frame_idx

frame_paths = sorted(frame_paths, key=sort_key)

base_filenames = []
seconds = []
for frame_path in frame_paths:
    base_filename = os.path.basename(frame_path).replace(".jpg", "")
    second = base_filename.split("_")[-2]
    base_filenames.append(base_filename)
    seconds.append(second)

new_df = pd.DataFrame({"base_filename": base_filenames, "second": seconds})
new_df.to_csv("data/new_GT_annotated.csv", index=False)

## Dataset Statistics

In [34]:
import pandas as pd

# df = pd.read_csv("data/GT_fully_annotated.csv")
# df = pd.read_csv("data/new_GT_annotated.csv")
df = pd.read_csv("data/GT_merged.csv")

ignore_basefilenames = [
    "clipped_1_14_Semiautomatic_nail_gun_accidents",
    "clipped_0_13_overhead_drilling_v1",
    "clipped_0_15_Drill work for roof ceiling shorts viral youtubeshortvideo roof",
    "clipped_0_22_fall ceiling drilling videoshot",
    "clipped_165_203_How to FRAME a Wall - 3 EASY STEPS_1080p",
    "clipped_0_19_Hand taping inside corners is always fun! LEVEL5 knifes always make the job easier",
    "clipped_0_25_Finishing Drywall Butt Joints with LEVEL5 Hand Tools",
    "clipped_0_11_Hard working Malaysia constructionworker",
    "clipped_0_12_Amazing fastest work rebar tying skill",
    "clipped_0_15_GUARANTEE",
    "clipped_11_25_Rebar tying (2)"
]

df = df[~df["base_filename"].isin(ignore_basefilenames)].reset_index(drop=True)

annotation_cols = ["action_1", "action_2", "action_3"]
annotation_cols = [col for col in annotation_cols if col in df.columns]

all_annotations = (
    df[annotation_cols]
    .stack()                  # flatten the three columns into one
    .astype(str)
    .str.strip()             # remove leading/trailing whitespace
)

# remove empty strings / nan
all_annotations = all_annotations[
    (all_annotations != "") &
    (all_annotations.str.lower() != "nan")
]

# count occurrences
annotation_counts = all_annotations.value_counts()

# print results
print("=== Kinds of Annotations ===")
print(annotation_counts)

print(f"\nTotal number of unique annotations: {annotation_counts.shape[0]}")

num_videos = df["base_filename"].nunique()
print(f"Number of unique videos: {num_videos}") #NOTE: Why 489? not 100?
num_rows = len(df)
print(f"Number of rows: {num_rows}")

=== Kinds of Annotations ===
none                              1251
lay brick/block                    336
drill                              315
plaster                            278
tie rebar                          249
lift/carry sheets                  243
apply adhesive/mortar              205
install tiles                      194
lift/carry window                  168
install roofing sheets             164
lift/carry cement bags             158
climb/climb down ladders/steps     133
screed                             126
attach rigging                     105
lift/carry door                     91
drive compactor                     85
measure dimensions/distances        85
weld                                80
paint                               73
hammer                              67
drive vehicles                      66
lift/carry hose                     60
measure the level                   59
push/pull cart                      55
sweep                              

In [32]:
action_cols = ["action_1", "action_2", "action_3"]

unique_action_counts = (
    df.melt(
        id_vars=["base_filename"],
        value_vars=action_cols,
        value_name="action"
    )
    .dropna(subset=["action"])
    .groupby("base_filename")["action"]
    .nunique()
    .reset_index(name="num_unique_actions")
)

avg_unique_actions = unique_action_counts["num_unique_actions"].mean()

print(f"Average number of unique actions per video: {avg_unique_actions:.2f}")

Average number of unique actions per video: 2.44


In [ ]:
# df1 = pd.read_csv("data/GT_fully_annotated.csv")
df1 = pd.read_csv("data/GT_fully_annotated_jinsik.csv")
df2 = pd.read_csv("data/GT_fully_annotated_seongju.csv")
# 건우-성주 98.33 건우-진식 89.09 진식-성주 89.09

action_cols = ["action_1", "action_2", "action_3"]

# Match rows by video and second
merged = pd.merge(
    df1,
    df2,
    on=["base_filename", "second"],
    suffixes=("_rater1", "_rater2"),
    how="inner"
)


def get_action_set(row, suffix):
    actions = {
        row[f"{col}_{suffix}"]
        for col in action_cols
        if pd.notna(row[f"{col}_{suffix}"])
    }
    
    # Remove whitespace just in case
    return {str(action).strip() for action in actions}


# def check_agreement(row):
#     actions_1 = get_action_set(row, "rater1")
#     actions_2 = get_action_set(row, "rater2")

#     # Agree if at least one action overlaps
#     return int(len(actions_1 & actions_2) > 0)

def check_agreement(row):
    actions_1 = get_action_set(row, "rater1")
    actions_2 = get_action_set(row, "rater2")

    # Agree only when the entire action sets are identical
    return int(actions_1 == actions_2)


merged["agree"] = merged.apply(check_agreement, axis=1)


# Agreement ratio
agreement_ratio = merged["agree"].mean()

print(f"Matched rows: {len(merged)}")
print(f"Agreed rows: {merged['agree'].sum()}")
print(f"Not agreed rows: {(merged['agree'] == 0).sum()}")
print(f"Agreement ratio: {agreement_ratio:.4f}")
print(f"Agreement percentage: {agreement_ratio * 100:.2f}%")
avg_agreement_ratio = (89.09*2+98.33)/3
print(f"Avg agreement ratio: {avg_agreement_ratio:.2f}%")

Matched rows: 4309
Agreed rows: 3839
Not agreed rows: 470
Agreement ratio: 0.8909
Agreement percentage: 89.09%


In [43]:
(89.09*2+98.33)/3

92.17

In [8]:
df = pd.read_csv("data/GT_fully_annotated.csv")
action_cols = ["action_1", "action_2", "action_3"]

# Convert action columns from wide format to long format
actions = df.melt(
    id_vars=["base_filename"],
    value_vars=action_cols,
    value_name="action_class"
)

# Remove empty annotations
actions = actions.dropna(subset=["action_class"])

# Each action is counted only once per video
actions_unique = actions.drop_duplicates(subset=["base_filename", "action_class"])

# Count the number of unique videos containing each action
action_counts = (
    actions_unique
    .groupby("action_class")["base_filename"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="video_count")
)

print(action_counts)

                      action_class  video_count
0                             none           84
1                            drill           19
2   climb/climb down ladders/steps           16
3            apply adhesive/mortar           12
4                lift/carry window           12
5                lift/carry sheets            9
6                        tie rebar            8
7                  lay brick/block            8
8                          plaster            7
9                           screed            6
10                   install tiles            5
11                  push/pull cart            5
12                           paint            5
13    measure dimensions/distances            4
14                          hammer            4
15          install roofing sheets            4
16                  attach rigging            3
17                           sweep            3
18                 drive compactor            3
19          lift/carry cement bags      

In [39]:
df = pd.read_csv("data/GT_unique_basefilenames.csv")
# ratio of value in "uniformat"
df["uniformat"].value_counts(normalize=True).sort_values(ascending=False)

uniformat
shell                0.38
interiors            0.34
substructure         0.20
building sitework    0.08
Name: proportion, dtype: float64

## Gemini to GPT

In [5]:
# gemini_path = "output/inference_results_gemini-3.1-pro-preview_videos.json"
gemini_path = "output/inference_results_gemini-3.5-flash-lite_videos.json"
org_gemini_filename = os.path.basename(gemini_path).replace(".json", "")

gpt_path = "output/inference_results_gpt-5.6-terra.json"
out_path = f"output/{org_gemini_filename}_gpt_format.json"
# out_path = "output/inference_results_gemini-3.1-flash-lite-preview_video_gpt_format.json"

gemini_rows = utils.read_jsonl(gemini_path)
gpt_rows = utils.read_jsonl(gpt_path)

gemini_by_key = {
    data_utils.video_key_from_video_path(r["video_path"]): r
    for r in gemini_rows
}
gpt_by_key = defaultdict(list)

for r in gpt_rows:
    key, sec, frame_idx = data_utils.parse_frame_path(r["frame_path"])
    rr = dict(r)
    rr["_video_key"] = key
    rr["_second"] = sec
    rr["_frame_idx"] = frame_idx
    gpt_by_key[key].append(rr)

converted = []
missing_keys = set()

for key, frames in gpt_by_key.items():
    gem = gemini_by_key.get(key)
    if gem is None:
        missing_keys.add(key)
        continue

    n = len(frames)
    per_frame_cost = gem.get("inference_cost", 0) / n if n > 0 else 0
    per_frame_time = gem.get("inference_time", 0) / n if n > 0 else 0

    for fr in sorted(frames, key=lambda x: (x["_second"], x["_frame_idx"])):
        converted.append({
            "frame_path": fr["frame_path"],
            "action": data_utils.action_at_second(gem.get("segments", []), fr["_second"]),
            "inference_cost": per_frame_cost,
            "inference_time": per_frame_time,
        })

with open(out_path, "w", encoding="utf-8") as f:
    for row in converted:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("gemini videos:", len(gemini_rows))
print("gpt frames:", len(gpt_rows))
print("converted frames:", len(converted))
print("missing video keys:", len(missing_keys))
print(sorted(list(missing_keys))[:10])
print("output:", out_path)
print(converted[:3])

gemini videos: 100
gpt frames: 4517
converted frames: 4517
missing video keys: 0
[]
output: output/inference_results_gemini-3.5-flash-lite_videos_gpt_format.json
[{'frame_path': 'data/frames_fps1\\clipped_0_10_NYC_crane_and__rigging_rebar_bundles_team_safety_always_firstcomment_if_you_do_construction_0_17.jpg', 'action': 'attach rigging', 'inference_cost': 6.882222222222222e-05, 'inference_time': 0.48172911008199054}, {'frame_path': 'data/frames_fps1\\clipped_0_10_NYC_crane_and__rigging_rebar_bundles_team_safety_always_firstcomment_if_you_do_construction_1_47.jpg', 'action': 'attach rigging', 'inference_cost': 6.882222222222222e-05, 'inference_time': 0.48172911008199054}, {'frame_path': 'data/frames_fps1\\clipped_0_10_NYC_crane_and__rigging_rebar_bundles_team_safety_always_firstcomment_if_you_do_construction_2_77.jpg', 'action': 'attach rigging', 'inference_cost': 6.882222222222222e-05, 'inference_time': 0.48172911008199054}]
